# Week 4: Your First Classifier: K Nearest Neighbors

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: Name that penguin

Scientists measured hundreds of penguins on three islands in Antarctica. If a new penguin has a 45 mm bill and 210 mm flippers, which species is it? You will train a model that answers this in seconds.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

In [ ]:
penguins = load("penguins.csv").dropna()
print(penguins["species"].value_counts())
penguins.head()

## Teach 1: Seeing the classes

Each species clusters in a different area of the chart. That is the pattern a classifier learns.

In [ ]:
colors = {"Adelie": "#f7941d", "Chinstrap": "#39b54a", "Gentoo": "#2271b1"}
for species, group in penguins.groupby("species"):
    plt.scatter(group["bill_length_mm"], group["flipper_length_mm"], label=species, color=colors[species])
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.title("Three penguin species")
plt.legend()
plt.show()

## Teach 2: K nearest neighbors

**K nearest neighbors (KNN)** is the friendliest classifier: to label a new point, look at the **k closest** points you already know and take a vote. If 4 of the 5 nearest penguins are Gentoo, it is probably Gentoo.

**Concept checkpoint:** look at the chart. A penguin with bill 45 and flipper 210. Which color are its neighbors?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

features = ["bill_length_mm", "flipper_length_mm"]
X = penguins[features]
y = penguins["species"]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X, y)

new_penguin = pd.DataFrame({"bill_length_mm": [45], "flipper_length_mm": [210]})
print("Predicted species:", knn.predict(new_penguin)[0])

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Predict the species for three new penguins: (39, 185), (49, 195), and (47, 220). Check each answer against the chart. Do they make sense?

### Medium
Train a second KNN using `bill_depth_mm` and `body_mass_g` instead. Plot those two features colored by species. Is this pair easier or harder to separate?

### Spicy
**Unsupervised learning.** Use `KMeans(n_clusters=3)` on the two original features with **no labels**. Plot the clusters it found next to the real species. How close did it get without ever seeing a label?

In [ ]:
# MILD: three new penguins
new_birds = pd.DataFrame({
    "bill_length_mm":    [39, 49, 47],
    "flipper_length_mm": [185, 195, 220],
})
for i, species in enumerate(knn.predict(new_birds)):
    print("Penguin", i + 1, "is predicted to be", species)

In [ ]:
# MEDIUM: a different pair of features
features2 = ["bill_depth_mm", "body_mass_g"]
for species, group in penguins.groupby("species"):
    plt.scatter(group["bill_depth_mm"], group["body_mass_g"], label=species, color=colors[species])
plt.xlabel("Bill depth (mm)")
plt.ylabel("Body mass (g)")
plt.legend()
plt.show()

knn2 = KNeighborsClassifier(n_neighbors=5)
knn2.fit(penguins[features2], y)
print(knn2.predict(pd.DataFrame({"bill_depth_mm": [18], "body_mass_g": [4000]}))[0])
# Adelie and Chinstrap overlap a lot on this pair, so it is harder to separate.

In [ ]:
# SPICY: clustering with no labels
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
clusters = kmeans.fit_predict(X)

plt.scatter(X["bill_length_mm"], X["flipper_length_mm"], c=clusters, cmap="viridis")
plt.title("Groups found with NO labels")
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.show()

print(pd.crosstab(y, clusters))   # rows: real species, columns: cluster number

## Extra activities (if you finish early)

- Change `n_neighbors` to 1 and then to 50. Predict the same penguin each time. Does the answer change?
- Use all four measurements as features. Does the prediction for the Hook penguin change?
- Draw on paper: a new point surrounded by 3 orange and 2 blue neighbors. What does k = 5 predict? What about k = 3?

## Reflection

- Explain KNN to a younger student in one sentence.
- What is the difference between **supervised** (KNN) and **unsupervised** (KMeans) learning?